# Random walk and diffusion

*thermodynamics · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/thermodynamics/random_walk_diffusion)

Thousands of Brownian particles each take random kicks. Stored as one (time, walker, 2) tensor, the walk is a single cumsum, and the mean-squared displacement $\langle x_i x_i\rangle$ — one einsum — grows linearly in time as $2dDt$, while the cloud of walkers spreads out as a Gaussian.

**Vector form**

$$\mathbf x(t + \Delta t) = \mathbf x(t) + \sqrt{2D\Delta t}\;\boldsymbol\xi,\qquad \langle \lVert\mathbf x\rVert^2 \rangle = 2dDt,\qquad \rho(\mathbf x, t) = \frac{e^{-\lVert\mathbf x\rVert^2 / 4Dt}}{4\pi D t}$$

**Index (tensor) form**

$$x_{tni} = \sum_{s \le t} \sqrt{2D\Delta t}\;\xi_{sni},\qquad \mathrm{MSD}_t = \frac1N\,x_{tni}\,x_{tni} = 2dD\,t_t$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\xi_{sni}$ | `torch.randn(S, N, 2)` | (S, N, 2) | independent standard-normal kick for step $s$, walker $n$, component $i$ |
| $x_{tni}$ | `cumsum(steps, dim=0)` | (T, N, 2) | position of walker $n$ at time $t$: the running sum of its kicks |
| $x_{tni}x_{tni}$ | `einsum('tni,tni->t', x, x) / N` | (T,) | repeated $n$ and $i$ ⇒ summed: mean-squared displacement at each time |
| $D_{\text{fit}}$ | `einsum('t,t->', t, msd) / (2d einsum('t,t->', t, t))` | scalar | least-squares slope of MSD vs $t$ through the origin, divided by $2d$ |
| $\rho(\mathbf x, t)$ | `gaussian_density(X, Y, t, D)` | (G, G) | solution of the diffusion equation $\partial_t\rho = D\,\partial_i\partial_i\rho$ from a point source |

Each step is a Gaussian kick of variance $2D\Delta t$ per component, so after time $t$ every coordinate has variance $2Dt$ and the squared distance from the start averages $2dDt$ in $d$ dimensions (here $d = 2$). Summing over the walker index $n$ turns the random cloud into a smooth, predictable law — that is diffusion. The finite number of walkers makes the measured MSD wobble by roughly $\sqrt{2/N}$ relative; the seed is fixed so the picture is repeatable. Units are arbitrary (µm and s, say).


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/thermodynamics/random_walk_diffusion/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    D, n_steps = params["D"], params["n_steps"]
    t = tensors.times(params["t_max"], n_steps, dev)  # (T,)
    dt = (t[1] - t[0]).item()
    kicks = tensors.random_kicks(n_steps, params["n_walkers"], D, dt, dev)  # (S, N, 2)

    r = solve.solve(t, kicks)
    var = r["x"][-1].var(dim=0)  # (2,)  spread of the final cloud along x and y
    print(f"fitted D = {r['D_fit']:.3f}   true D = {D:.3f}")
    print(f"final variance per axis {var.tolist()}   theory 2Dt = {2 * D * t[-1]:.2f}")
    return plot.animate(t, r["x"], r["msd"], D, r["D_fit"], out_dir / "random_walk_diffusion.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/thermodynamics/random_walk_diffusion/equation.py
r"""Physics → tensor translation.

Brownian motion: N particles n = 1..N in d = 2 dimensions (i ∈ {x, y}) each receive a random kick every
time step Δt,

    x_{t+1,ni} = x_{t,ni} + \sqrt{2 D \Delta t} \; \xi_{t,ni},       \xi \sim \mathcal N(0, 1)

so the position is the running sum of the kicks, x_{tni} = \sum_{s \le t} \sqrt{2D\Delta t}\,\xi_{sni}.

The mean-squared displacement averages the squared distance over all walkers. With the repeated indices
n and i summed:

    \mathrm{MSD}_t = \frac1N \, x_{tni} \, x_{tni} = 2 d D \, t_t        (einsum('tni,tni->t') / N)

Fitting a straight line through the origin, D_fit = t_t \mathrm{MSD}_t / (2d \, t_t t_t), sums over t.
The density of walkers solves the diffusion equation ∂_t ρ = D ∂_i ∂_i ρ:

    \rho(\mathbf x, t) = \frac{1}{4 \pi D t} \, e^{-x_i x_i / 4 D t}
"""

import math

import torch


def kick_size(D: float, dt: float) -> float:
    """Standard deviation of one step along each axis: √(2 D Δt)."""
    return math.sqrt(2 * D * dt)


def mean_squared_displacement(x: torch.Tensor) -> torch.Tensor:
    r"""MSD_t = (1/N) x_{tni} x_{tni}, summed over walkers n and components i.   (T, N, d) → (T,)"""
    return torch.einsum("tni,tni->t", x, x) / x.shape[1]


def msd_theory(t: torch.Tensor, D: float, d: int = 2) -> torch.Tensor:
    """Einstein's result ⟨x_i x_i⟩ = 2 d D t.   (T,) → (T,)"""
    return 2 * d * D * t


def fit_diffusion(t: torch.Tensor, msd: torch.Tensor, d: int = 2) -> float:
    """Least-squares slope through the origin, divided by 2d:  D = Σ_t t·MSD / (2d Σ_t t²)."""
    return (torch.einsum("t,t->", t, msd) / (2 * d * torch.einsum("t,t->", t, t))).item()


def gaussian_density(X: torch.Tensor, Y: torch.Tensor, t: float, D: float) -> torch.Tensor:
    """ρ(x, y, t) = exp(−(x² + y²) / 4Dt) / (4πDt): walkers per unit area, all started at the origin."""
    return torch.exp(-(X**2 + Y**2) / (4 * D * t)) / (4 * math.pi * D * t)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/thermodynamics/random_walk_diffusion/tensors.py
"""Build the concrete tensors: the time grid and every random kick of every walker."""

import torch

from .equation import kick_size

DIM = 2  # walkers move in the plane


def times(t_max: float, n_steps: int, device: torch.device) -> torch.Tensor:
    """t_t = 0, Δt, 2Δt, ..., t_max.  Shape (S + 1,)."""
    return torch.linspace(0, t_max, n_steps + 1, device=device)


def random_kicks(n_steps: int, n_walkers: int, D: float, dt: float, device: torch.device, seed: int = 0):
    """√(2DΔt) ξ_{sni} with ξ standard normal.  Shape (S, N, 2).  Fixed seed → same picture every run."""
    gen = torch.Generator().manual_seed(seed)
    xi = torch.randn(n_steps, n_walkers, DIM, generator=gen)  # (S, N, 2)
    return (kick_size(D, dt) * xi).to(device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/thermodynamics/random_walk_diffusion/solve.py
"""Add up the kicks into paths and measure how fast the cloud spreads."""

import torch

from .equation import fit_diffusion, mean_squared_displacement


def solve(t: torch.Tensor, kicks: torch.Tensor) -> dict:
    """t: (T,), kicks: (T-1, N, 2).  Returns positions (T, N, 2), MSD (T,) and the fitted D."""
    start = torch.zeros_like(kicks[:1])  # (1, N, 2)  everyone starts at the origin
    x = torch.cumsum(torch.cat([start, kicks]), dim=0)  # (T, N, 2)  x_t = sum of kicks up to t

    msd = mean_squared_displacement(x)  # (T,)
    return {"x": x, "msd": msd, "D_fit": fit_diffusion(t, msd, x.shape[-1])}


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/thermodynamics/random_walk_diffusion/plot.py
"""Animate the cloud of walkers over the Gaussian density, next to the growing MSD and Einstein's line."""

from pathlib import Path

import matplotlib.pyplot as plt
import torch

from problems._common.plotting import ACCENT, frame_indices, save_gif

from .equation import gaussian_density, msd_theory

N_SHOWN = 400  # dots drawn (all walkers are used for the MSD)
N_TRAILS = 4  # walkers whose whole path is drawn


def animate(t: torch.Tensor, x: torch.Tensor, msd: torch.Tensor, D: float, D_fit: float, path: Path) -> Path:
    times = t.cpu().numpy()
    pos = x[:, :N_SHOWN].cpu().numpy()  # (T, N_SHOWN, 2)
    msd_np = msd.cpu().numpy()
    half = 3.0 * (2 * D * times[-1]) ** 0.5  # ±3 standard deviations at the end
    grid = torch.linspace(-half, half, 120)
    X, Y = torch.meshgrid(grid, grid, indexing="xy")  # (G, G)

    fig, (ax, ax_m) = plt.subplots(1, 2, figsize=(7, 4))
    density = ax.imshow(
        torch.zeros(120, 120).numpy(), extent=(-half, half, -half, half), origin="lower", cmap="Blues", vmin=0
    )
    dots = ax.scatter(pos[0, :, 0], pos[0, :, 1], s=2, color="black", alpha=0.5)
    trails = [ax.plot([], [], lw=1, color=ACCENT[1 + k])[0] for k in range(N_TRAILS)]
    ax.set_xlim(-half, half)
    ax.set_ylim(-half, half)
    ax.set_aspect("equal")
    ax.grid(False)
    ax.set_xlabel("x (µm)")
    ax.set_ylabel("y (µm)")
    clock = ax.set_title("")

    ax_m.plot(times, msd_theory(t, D).cpu().numpy(), color="gray", ls="--", label=f"2dDt  (D = {D:.2f})")
    msd_line = ax_m.plot([], [], color=ACCENT[0], lw=2, label=f"simulation  (fit D = {D_fit:.2f})")[0]
    ax_m.set_xlim(0, times[-1])
    ax_m.set_ylim(0, 1.15 * max(msd_np.max(), 4 * D * times[-1]))
    ax_m.set_xlabel("t (s)")
    ax_m.set_ylabel("⟨x_i x_i⟩ (µm²)")
    ax_m.set_title("mean-squared displacement")
    ax_m.legend(fontsize=8, loc="upper left")

    def update(step: int):
        if step > 0:  # the density is a point (delta function) at t = 0
            rho = gaussian_density(X, Y, float(times[step]), D)
            density.set_data(rho.numpy())
            density.set_clim(0, rho.max().item())
        dots.set_offsets(pos[step])
        for k, trail in enumerate(trails):
            trail.set_data(pos[: step + 1, k, 0], pos[: step + 1, k, 1])
        msd_line.set_data(times[: step + 1], msd_np[: step + 1])
        clock.set_text(f"t = {times[step]:.1f} s")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(times), 72), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_walkers': 2000,
    'D': 1.0,
    't_max': 10.0,
    'n_steps': 200
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.thermodynamics.random_walk_diffusion")]:
    del sys.modules[name]
main = importlib.import_module("problems.thermodynamics.random_walk_diffusion.main")
problem_dir = Path("problems/thermodynamics/random_walk_diffusion")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
